# Advanced Spark Transformations

In [1]:
import os
import sys
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, IntegerType
)

# Python versiya uyğunsuzluğunu qarşısını almaq üçün mühit dəyişənləri
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# 1. SparkSession-ın yaradılması
spark = (
    SparkSession.builder
    .appName("Lesson21-DataFrames-SparkSQL")
    .master("local[*]")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "matrix")
    .config("spark.hadoop.fs.s3a.secret.key", "matrix123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate()
)

In [6]:
customers_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("customer_name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("balance", DoubleType(), True),
    StructField("vip_status", StringType(), True)
])

df_customers = spark.read.schema(customers_schema).option("header", "true").csv("s3a://spark-lesson-3/customers.csv")
df_card_trn = spark.read.option("header", "true").option("inferSchema", "true").csv("s3a://spark-lesson-3/card_trn.csv")
df_trn_items = spark.read.option("multiline", "true").json("s3a://spark-lesson-3/trn_items.json")

df_customers.show(3)
df_card_trn.show(3)
df_trn_items.show(3, truncate=False)

+-----------+---------------+---+-------+----------+
|customer_id|  customer_name|age|balance|vip_status|
+-----------+---------------+---+-------+----------+
|       C001|  Rashad Aliyev| 23|1230.15|         Y|
|       C002|Sabina Mammadov| 60|3651.47|         Y|
|       C003|Sevinj Karimova| 19|   NULL|         Y|
+-----------+---------------+---+-------+----------+
only showing top 3 rows

+------+-----------+------+----------+--------+
|trn_id|customer_id|amount|  trn_date|merchant|
+------+-----------+------+----------+--------+
| T0001|       C001| 67.14|2026-03-09|   SOCAR|
| T0002|       C001|265.14|2026-07-11| Ecoteks|
| T0003|       C002|448.81|2026-02-26|    Wolt|
+------+-----------+------+----------+--------+
only showing top 3 rows

+-------------------------------------------------------+------+
|items                                                  |trn_id|
+-------------------------------------------------------+------+
|[{Cheese, 1, 5.5}, {Charger, 3, 12.5}, {Bread, 

In [7]:
(
    df_customers
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .parquet("s3a://spark-lesson-3/bronze/customers")
)

(
    df_card_trn
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .parquet("s3a://spark-lesson-3/bronze/card_trn")
)

(
    df_trn_items
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .parquet("s3a://spark-lesson-3/bronze/trn_items")
)

print("Bütün xam datalar 'bronze' qatına uğurla yazıldı!")

Bütün xam datalar 'bronze' qatına uğurla yazıldı!


In [8]:
df_customers_clean = (
    spark.read.parquet("s3a://spark-lesson-3/bronze/customers")
    .fillna({"balance": 0.0})
    .withColumn(
        "age_group",
        F.when(F.col("age") < 25, "young")
         .when((F.col("age") >= 25) & (F.col("age") <= 40), "adult")
         .otherwise("senior")
    )
    .withColumnRenamed("vip_status", "flg_is_vip")
    .withColumn("insert_date", F.current_date())
)

df_card_trn_clean = (
    spark.read.parquet("s3a://spark-lesson-3/bronze/card_trn")
    .withColumn("trn_month", F.date_format(F.col("trn_date"), "yyyy-MM"))
    .withColumn("amount", F.round(F.col("amount"), 2))
)

df_customers_clean.show(3, truncate=False)
df_card_trn_clean.show(3, truncate=False)

+-----------+---------------+---+-------+----------+---------+-----------+
|customer_id|customer_name  |age|balance|flg_is_vip|age_group|insert_date|
+-----------+---------------+---+-------+----------+---------+-----------+
|C001       |Rashad Aliyev  |23 |1230.15|Y         |young    |2026-09-13 |
|C002       |Sabina Mammadov|60 |3651.47|Y         |senior   |2026-09-13 |
|C003       |Sevinj Karimova|19 |0.0    |Y         |young    |2026-09-13 |
+-----------+---------------+---+-------+----------+---------+-----------+
only showing top 3 rows

+------+-----------+------+----------+--------+---------+
|trn_id|customer_id|amount|trn_date  |merchant|trn_month|
+------+-----------+------+----------+--------+---------+
|T0001 |C001       |67.14 |2026-03-09|SOCAR   |2026-03  |
|T0002 |C001       |265.14|2026-07-11|Ecoteks |2026-07  |
|T0003 |C002       |448.81|2026-02-26|Wolt    |2026-02  |
+------+-----------+------+----------+--------+---------+
only showing top 3 rows



In [10]:
from pyspark.sql.functions import broadcast

df_inner_joined = df_customers_clean.join(df_card_trn_clean, on="customer_id", how="inner")

df_no_trn = df_customers_clean.join(df_card_trn_clean, on="customer_id", how="left_anti")

print(f"Tranzaksiyası olan sətir sayı: {df_inner_joined.count()}")
print(f"Tranzaksiyası olmayan müştəri sayı: {df_no_trn.count()}")

df_broadcast_joined = df_customers_clean.join(broadcast(df_card_trn_clean), on="customer_id", how="inner")
df_broadcast_joined.explain()

Tranzaksiyası olan sətir sayı: 45
Tranzaksiyası olmayan müştəri sayı: 3
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [customer_id#452, customer_name#453, age#454, balance#467, flg_is_vip#481, age_group#474, 2026-09-13 AS insert_date#488, trn_id#496, amount#514, trn_date#499, merchant#500, trn_month#506]
   +- BroadcastHashJoin [customer_id#452], [customer_id#497], Inner, BuildRight, false
      :- Project [customer_id#452, customer_name#453, age#454, coalesce(nanvl(balance#455, null), 0.0) AS balance#467, vip_status#456 AS flg_is_vip#481, CASE WHEN (age#454 < 25) THEN young WHEN ((age#454 >= 25) AND (age#454 <= 40)) THEN adult ELSE senior END AS age_group#474]
      :  +- Filter isnotnull(customer_id#452)
      :     +- FileScan parquet [customer_id#452,customer_name#453,age#454,balance#455,vip_status#456] Batched: true, DataFilters: [isnotnull(customer_id#452)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[s3a://spark-lesson-3/bronze/customers], Partitio

In [16]:
window_spec = Window.partitionBy("customer_id").orderBy(F.col("amount").desc())

df_top2_trn = (
    df_card_trn_clean
    .withColumn("rn", F.row_number().over(window_spec))
    .filter(F.col("rn") <= 2)
    .drop("rn")
)

df_top2_trn.show(5)

df_customer_agg = (
    df_inner_joined
    .groupBy("customer_id", "customer_name", "age_group", "flg_is_vip")
    .agg(
        F.count("trn_id").alias("total_transactions"),
        F.sum("amount").alias("total_amount"),
        F.avg("amount").alias("avg_amount"),
        F.max("amount").alias("max_amount")
    )
    .orderBy(F.col("total_amount").desc())
)

df_customer_agg.show(5, truncate=False)

+------+-----------+------+----------+--------+---------+
|trn_id|customer_id|amount|  trn_date|merchant|trn_month|
+------+-----------+------+----------+--------+---------+
| T0002|       C001|265.14|2026-07-11| Ecoteks|  2026-07|
| T0001|       C001| 67.14|2026-03-09|   SOCAR|  2026-03|
| T0003|       C002|448.81|2026-02-26|    Wolt|  2026-02|
| T0004|       C002|224.61|2026-05-11|   Bravo|  2026-05|
| T0009|       C003|325.72|2026-06-24|   SOCAR|  2026-06|
+------+-----------+------+----------+--------+---------+
only showing top 5 rows

+-----------+---------------+---------+----------+------------------+------------+------------------+----------+
|customer_id|customer_name  |age_group|flg_is_vip|total_transactions|total_amount|avg_amount        |max_amount|
+-----------+---------------+---------+----------+------------------+------------+------------------+----------+
|C003       |Sevinj Karimova|young    |Y         |6                 |1471.87     |245.31166666666664|325.72    |
|

In [17]:
df_trn_with_items = df_card_trn_clean.join(df_trn_items, on="trn_id", how="inner")

df_exploded = (
    df_trn_with_items
    .withColumn("item", F.explode("items"))
    .select(
        "trn_id",
        "amount",
        F.col("item.product_name").alias("product_name"),
        F.col("item.qty").alias("qty"),
        F.col("item.unit_price").alias("unit_price")
    )
    .withColumn("line_total", F.col("qty") * F.col("unit_price"))
)

df_comparison = (
    df_exploded
    .groupBy("trn_id", "amount")
    .agg(F.sum("line_total").alias("calculated_total"))
    .withColumn("difference", F.abs(F.col("amount") - F.col("calculated_total")))
)

df_comparison.show(5)

+------+------+----------------+------------------+
|trn_id|amount|calculated_total|        difference|
+------+------+----------------+------------------+
| T0001| 67.14|            44.2|22.939999999999998|
+------+------+----------------+------------------+



In [18]:
df_pivot = (
    df_inner_joined
    .groupBy("customer_id")
    .pivot("trn_month")
    .agg(F.sum("amount"))
    .na.fill(0)
)

df_pivot.show(10, truncate=False)

+-----------+-------+-------+-------+-----------------+-------+-------+-------+-------+
|customer_id|2026-01|2026-02|2026-03|2026-04          |2026-05|2026-06|2026-07|2026-08|
+-----------+-------+-------+-------+-----------------+-------+-------+-------+-------+
|C006       |0.0    |0.0    |0.0    |0.0              |0.0    |0.0    |0.0    |166.53 |
|C010       |0.0    |0.0    |0.0    |0.0              |251.5  |0.0    |0.0    |0.0    |
|C007       |442.89 |0.0    |0.0    |0.0              |0.0    |0.0    |0.0    |0.0    |
|C012       |641.71 |0.0    |0.0    |0.0              |0.0    |0.0    |0.0    |0.0    |
|C003       |0.0    |0.0    |198.47 |540.5699999999999|116.88 |325.72 |290.23 |0.0    |
|C015       |0.0    |282.38 |0.0    |0.0              |0.0    |0.0    |0.0    |0.0    |
|C004       |0.0    |0.0    |0.0    |0.0              |0.0    |0.0    |394.75 |0.0    |
|C011       |319.99 |0.0    |0.0    |0.0              |0.0    |0.0    |0.0    |0.0    |
|C014       |0.0    |0.0    |0.0

In [19]:
(
    df_customer_agg
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .parquet("s3a://spark-lesson-3/silver/customer_summary")
)

df_silver_read = spark.read.parquet("s3a://spark-lesson-3/silver/customer_summary")

df_silver_read.show(5, truncate=False)

+-----------+---------------+---------+----------+------------------+------------+------------------+----------+
|customer_id|customer_name  |age_group|flg_is_vip|total_transactions|total_amount|avg_amount        |max_amount|
+-----------+---------------+---------+----------+------------------+------------+------------------+----------+
|C003       |Sevinj Karimova|young    |Y         |6                 |1471.87     |245.31166666666664|325.72    |
|C016       |Gunel Mammadov |senior   |Y         |4                 |1471.15     |367.7875          |417.23    |
|C013       |Orkhan Mammadov|adult    |Y         |6                 |1445.7      |240.95000000000002|421.3     |
|C005       |Vusal Guliyev  |young    |N         |5                 |1117.71     |223.542           |429.31    |
|C008       |Aysel Mammadov |young    |N         |5                 |1106.85     |221.36999999999998|418.37    |
+-----------+---------------+---------+----------+------------------+------------+--------------

In [20]:
spark.stop()